# Use_Model — EffoVPR Safetensors inference

Load config.json + model.safetensors, accept query/gallery file paths, retrieve Top 10 and rerank with local MNN.


## 1. Cài dependency

Giữ đúng các phiên bản chính đã dùng trong notebook train để giảm nguy cơ lệch kiến trúc Hugging Face/PyTorch.

In [ ]:
%pip install -q transformers==5.18.0 datasets==5.0.1 faiss-cpu==1.15.1 "PyYAML>=6.0" "pandas>=2.0" "Pillow>=10.0" "matplotlib>=3.7" "tqdm>=4.9" "safetensors>=0.4.3"


In [ ]:
from pathlib import Path
import os
import sys
import json
import hashlib
import io
import math
import time

NOTEBOOK_NAME = "Use_Model"
WORK_DIR = Path("/kaggle/working") / NOTEBOOK_NAME if Path("/kaggle/working").exists() else Path.cwd() / NOTEBOOK_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

for folder in (
    "src/effovpr/models",
    "src/effovpr/utils",
    "src/effovpr/reranking",
    "artifacts/features",
    "artifacts/results",
    "artifacts/cache/huggingface/models",
    "artifacts/cache/huggingface/datasets",
):
    (WORK_DIR / folder).mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(WORK_DIR / "artifacts/cache/huggingface")
os.environ["HF_HUB_CACHE"] = str(WORK_DIR / "artifacts/cache/huggingface/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORK_DIR / "artifacts/cache/huggingface/datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Workspace:", WORK_DIR)

## Model and image paths

Set MODEL_DIR, QUERY_IMAGE_PATH and GALLERY_DIR. Architecture and inference settings come from config.json.


In [ ]:
MODEL_DIR = "/kaggle/input/your-model/artifacts/model"
QUERY_IMAGE_PATH = "/kaggle/input/your-query/query.jpg"
GALLERY_SOURCE = "folder"
GALLERY_DIR = "/kaggle/input/your-gallery"
GLOBAL_BATCH_SIZE = 4
RERANK_TOP_K = 10
SEED = 42


### `src/__init__.py`

In [ ]:
%%writefile src/__init__.py
"""EffoVPR package."""


### `src/effovpr/__init__.py`

In [ ]:
%%writefile src/effovpr/__init__.py
__all__ = ["EffoVPR", "CosFace", "build_effovpr_model"]

from .models.effovpr import EffoVPR
from .models.cosface import CosFace


def build_effovpr_model(*args, **kwargs):
    return EffoVPR(*args, **kwargs)


### `src/effovpr/models/cosface.py`

In [ ]:
%%writefile src/effovpr/models/cosface.py
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class CosFace(nn.Module):
    def __init__(self, in_features: int, out_features: int, scale: float = 30.0, margin: float = 0.4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings: torch.Tensor, targets: torch.Tensor):
        if targets is None:
            raise ValueError("CosFace requires target class indices")
        if embeddings.dim() != 2:
            embeddings = embeddings.view(embeddings.size(0), -1)
        if embeddings.size(1) != self.in_features:
            raise ValueError(f"Expected embedding dimension {self.in_features}, got {embeddings.size(1)}")
        if targets.ndim != 1 or targets.size(0) != embeddings.size(0):
            raise ValueError("targets must be a 1D tensor with one class index per embedding")
        if targets.dtype != torch.long:
            targets = targets.long()
        if targets.numel() and (targets.min() < 0 or targets.max() >= self.out_features):
            raise ValueError(f"targets must be in [0, {self.out_features})")
        norm_embeddings = F.normalize(embeddings, p=2, dim=1)
        norm_weight = F.normalize(self.weight, p=2, dim=1)
        cosine = norm_embeddings @ norm_weight.T
        target_mask = F.one_hot(targets, num_classes=self.out_features).to(dtype=cosine.dtype)
        logits = (cosine - self.margin * target_mask) * self.scale
        loss = F.cross_entropy(logits, targets)
        return logits, loss


### `src/effovpr/models/dino_attention.py`

In [ ]:
%%writefile src/effovpr/models/dino_attention.py
from __future__ import annotations

import math

import torch


def resolve_layer_index(layer, total_layers: int) -> int:
    if isinstance(layer, str):
        layer = layer.strip().lower()
        if layer.startswith("n-"):
            offset = int(layer.split("-")[-1])
            idx = total_layers - offset
        elif layer == "n":
            idx = total_layers - 1
        elif layer.startswith("n") and layer[1:].isdigit():
            idx = int(layer[1:]) - 1
        else:
            raise ValueError(f"Unsupported layer token: {layer}")
    else:
        idx = int(layer)
    if idx < 0:
        idx = total_layers + idx
    if idx < 0 or idx >= total_layers:
        raise ValueError(f"Layer index {idx} is out of range for total layers {total_layers}.")
    return idx


def parse_tokens(hidden_tokens: torch.Tensor):
    if hidden_tokens.ndim != 3:
        raise ValueError(f"Expected [batch, tokens, dim], got {tuple(hidden_tokens.shape)}")
    batch_size = hidden_tokens.size(0)
    cls_token = hidden_tokens[:, 0:1, :]
    register_tokens = hidden_tokens[:, 1:5, :] if hidden_tokens.size(1) > 5 else hidden_tokens[:, 1:1, :]
    patch_tokens = hidden_tokens[:, 5:, :]
    return cls_token, register_tokens, patch_tokens


def attention_patch_scores(q_patch: torch.Tensor, k_cls: torch.Tensor, temperature: float | None = None):
    if q_patch.ndim != 2:
        q_patch = q_patch.reshape(q_patch.size(0), -1)
    if temperature is None:
        temperature = math.sqrt(q_patch.size(-1))
    scores = (q_patch @ k_cls.reshape(-1)) / temperature
    return torch.softmax(scores, dim=-1)


def assert_patch_count(patch_tokens: torch.Tensor, image_resolution: int | None = None):
    patch_count = patch_tokens.shape[1]
    grid_h = int(round(math.sqrt(patch_count)))
    grid_w = patch_count // grid_h if grid_h > 0 else 0
    if image_resolution is not None:
        expected = (image_resolution // 14) ** 2
        if patch_count != expected:
            raise AssertionError(f"Expected patch count {expected} for resolution {image_resolution}, got {patch_count}")
    else:
        if grid_h * grid_w != patch_count:
            raise AssertionError(f"patch count mismatch: {patch_count} != {grid_h} * {grid_w}")


### `src/effovpr/models/dino_backbone.py`

In [ ]:
%%writefile src/effovpr/models/dino_backbone.py
from __future__ import annotations


import torch
from transformers import AutoConfig, AutoImageProcessor, AutoModel


class DINOv2Backbone(torch.nn.Module):
    def __init__(
        self,
        model_name: str = "facebook/dinov2-with-registers-large",
        with_registers: bool = True,
        cache_dir: str | None = None,
        backbone_config: dict | None = None,
        processor_config: dict | None = None,
    ):
        super().__init__()
        self.model_name = model_name
        self.with_registers = with_registers
        if backbone_config is None:
            self.model = AutoModel.from_pretrained(model_name, cache_dir=cache_dir)
            self.image_processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=cache_dir)
        else:
            config = dict(backbone_config)
            model_type = config.pop("model_type")
            self.model = AutoModel.from_config(AutoConfig.for_model(model_type, **config))
            from transformers import image_processing_utils
            import transformers
            processor = dict(processor_config)
            processor_type = processor.pop("image_processor_type")
            processor_class = getattr(transformers, processor_type)
            if not issubclass(processor_class, image_processing_utils.BaseImageProcessor):
                raise ValueError(f"Unsupported image processor: {processor_type}")
            self.image_processor = processor_class.from_dict(processor)
        self.hidden_dim = self.model.config.hidden_size
        self.patch_size = self.model.config.patch_size
        self.num_layers = len(self.encoder_layers())

    def encoder_layers(self):
        encoder = getattr(self.model, "encoder", None)
        if encoder is not None:
            layers = getattr(encoder, "layer", getattr(encoder, "layers", None))
            if layers is not None:
                return layers
        vit = getattr(self.model, "vit", None)
        encoder = getattr(vit, "encoder", None)
        if encoder is not None:
            layers = getattr(encoder, "layers", getattr(encoder, "layer", None))
            if layers is not None:
                return layers
        raise RuntimeError(f"Unsupported DINOv2 encoder structure for {type(self.model).__name__}")

    def forward(self, pixel_values: torch.Tensor, **kwargs):
        return self.model(pixel_values=pixel_values, output_hidden_states=True, return_dict=True, **kwargs)

    def configure_trainable_layers(self, last_n_layers: int = 5):
        if last_n_layers < 0 or last_n_layers > self.num_layers:
            raise ValueError(f"last_n_layers must be between 0 and {self.num_layers}")
        for param in self.model.parameters():
            param.requires_grad = False
        layers = self.encoder_layers()
        for layer in list(layers)[-last_n_layers:] if last_n_layers else []:
            for param in layer.parameters():
                param.requires_grad = True


### `src/effovpr/models/projection.py`

In [ ]:
%%writefile src/effovpr/models/projection.py
from __future__ import annotations

import torch
import torch.nn as nn


class GlobalProjection(nn.Module):
    def __init__(self, input_dim: int = 1024, output_dim: int = 1024):
        super().__init__()
        if output_dim not in (128, 256, 1024):
            raise ValueError("Supported global dimensions are 128, 256, and 1024.")
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.proj = nn.Linear(input_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


### `src/effovpr/models/effovpr.py`

In [ ]:
%%writefile src/effovpr/models/effovpr.py
from __future__ import annotations

from typing import Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from .cosface import CosFace
from .dino_attention import assert_patch_count, attention_patch_scores, parse_tokens, resolve_layer_index
from .dino_backbone import DINOv2Backbone
from .projection import GlobalProjection


class EffoVPR(nn.Module):
    def __init__(
        self,
        backbone_name: str = "facebook/dinov2-with-registers-large",
        global_dim: int = 1024,
        num_classes: Optional[int] = None,
        fine_tune_last_n_layers: int = 5,
        with_registers: bool = True,
        cosface_scale: float = 30.0,
        cosface_margin: float = 0.4,
        cache_dir: str | None = None,
        backbone_config: dict | None = None,
        processor_config: dict | None = None,
    ):
        super().__init__()
        self.backbone = DINOv2Backbone(model_name=backbone_name, with_registers=with_registers, cache_dir=cache_dir, backbone_config=backbone_config, processor_config=processor_config)
        self.global_dim = global_dim
        self.global_projection = GlobalProjection(input_dim=self.backbone.hidden_dim, output_dim=global_dim)
        self.num_classes = num_classes
        self.cosface = CosFace(in_features=global_dim, out_features=num_classes, scale=cosface_scale, margin=cosface_margin) if num_classes is not None else None
        self.configure_trainable_layers(last_n_layers=fine_tune_last_n_layers)

    def configure_trainable_layers(self, last_n_layers: int = 5):
        self.fine_tune_last_n_layers = last_n_layers
        if last_n_layers < 0 or last_n_layers > self.backbone.num_layers:
            raise ValueError(f"last_n_layers must be between 0 and {self.backbone.num_layers}")
        self.backbone.configure_trainable_layers(last_n_layers=last_n_layers)
        for param in self.global_projection.parameters():
            param.requires_grad = True
        if self.cosface is not None:
            for param in self.cosface.parameters():
                param.requires_grad = True

    def _prepare_images(self, images):
        if isinstance(images, torch.Tensor):
            return images
        if isinstance(images, list) and len(images) > 0 and isinstance(images[0], torch.Tensor):
            return torch.stack(images, dim=0)
        processed = self.backbone.image_processor(images=images, return_tensors="pt")
        return processed["pixel_values"]

    def _apply_projection(self, cls_token: torch.Tensor) -> torch.Tensor:
        feat = self.global_projection(cls_token)
        return F.normalize(feat, p=2, dim=-1)

    def extract_global(self, image):
        pixel_values = self._prepare_images(image)
        pixel_values = pixel_values.to(next(self.parameters()).device)
        outputs = self.backbone(pixel_values)
        hidden = outputs.last_hidden_state
        cls_token = hidden[:, 0, :]
        return self._apply_projection(cls_token)

    def extract_global_and_local(self, image, layer: int | str = "n-1"):
        pixel_values = self._prepare_images(image)
        pixel_values = pixel_values.to(next(self.parameters()).device)
        outputs = self.backbone(pixel_values)
        hidden_states = outputs.hidden_states
        layer_index = resolve_layer_index(layer, self.backbone.num_layers)
        selected = hidden_states[layer_index + 1]
        cls_token, register_tokens, patch_tokens = parse_tokens(selected)
        patch_tokens = patch_tokens.reshape(pixel_values.size(0), -1, patch_tokens.size(-1))
        assert_patch_count(patch_tokens, image_resolution=int(pixel_values.shape[-1]) if pixel_values.shape[-1] % self.backbone.patch_size == 0 else None)
        global_feat = self._apply_projection(outputs.last_hidden_state[:, 0, :])
        return global_feat, {"layer": layer_index, "cls_token": cls_token, "register_tokens": register_tokens, "patch_tokens": patch_tokens}

    def extract_qkv(self, image, layer: int | str = "n-1"):
        pixel_values = self._prepare_images(image)
        pixel_values = pixel_values.to(next(self.parameters()).device)
        outputs = self.backbone(pixel_values)
        hidden_states = outputs.hidden_states
        layer_index = resolve_layer_index(layer, self.backbone.num_layers)
        input_tokens = hidden_states[layer_index]
        block = self.backbone.encoder_layers()[layer_index]
        if hasattr(block, "norm1"):
            attention_input = block.norm1(input_tokens)
        elif hasattr(block, "layernorm_before"):
            attention_input = block.layernorm_before(input_tokens)
        else:
            raise RuntimeError(f"Cannot locate pre-attention normalization in {type(block).__name__}")
        attention = getattr(block, "attention", getattr(block, "self_attn", None))
        if attention is None:
            raise RuntimeError(f"Cannot locate attention module in {type(block).__name__}")
        # Hugging Face's DINOv2 implementations expose either q_proj/k_proj/v_proj
        # directly or query/key/value under an inner attention module.
        projections = [getattr(attention, name, None) for name in ("q_proj", "k_proj", "v_proj")]
        if all(projection is not None for projection in projections):
            q_proj, k_proj, v_proj = projections
        else:
            inner_attention = getattr(attention, "attention", attention)
            projections = [getattr(inner_attention, name, None) for name in ("query", "key", "value")]
            if not all(projection is not None for projection in projections):
                raise RuntimeError(f"Unsupported Q/K/V projection structure in {type(attention).__name__}")
            q_proj, k_proj, v_proj = projections
        q = q_proj(attention_input)
        k = k_proj(attention_input)
        v = v_proj(attention_input)
        cls_token, register_tokens, _ = parse_tokens(q)
        v_patch_tokens = parse_tokens(v)[2]
        assert_patch_count(v_patch_tokens, image_resolution=int(pixel_values.shape[-1]) if pixel_values.shape[-1] % self.backbone.patch_size == 0 else None)
        return {
            "q": q,
            "k": k,
            "v": v,
            "cls_token": cls_token,
            "register_tokens": register_tokens,
            "q_patch": parse_tokens(q)[2],
            "k_patch": parse_tokens(k)[2],
            "v_patch": parse_tokens(v)[2],
            "layer": layer_index,
        }

    def compute_attention_score(self, qkv: Dict[str, torch.Tensor], T1: float = 0.05):
        q_patch = qkv["q_patch"][0]
        k_cls = qkv["k"][0, 0, :]
        scores = attention_patch_scores(q_patch, k_cls)
        selected = scores > T1
        selected_v = qkv["v_patch"][0][selected]
        return scores, selected, selected_v

    def forward(self, images, labels=None, **kwargs):
        pixel_values = self._prepare_images(images)
        outputs = self.backbone(pixel_values)
        hidden = outputs.last_hidden_state
        cls_token = hidden[:, 0, :]
        global_feat = self._apply_projection(cls_token)
        if labels is not None and self.cosface is not None:
            logits, loss = self.cosface(global_feat, labels)
            return global_feat, logits, loss
        return global_feat


### `src/effovpr/reranking/mnn.py`

In [ ]:
%%writefile src/effovpr/reranking/mnn.py
from __future__ import annotations

import torch


def mutual_nearest_neighbor_matches(query_local: torch.Tensor, candidate_local: torch.Tensor, threshold: float = 0.65):
    query_local = torch.as_tensor(query_local, dtype=torch.float32)
    candidate_local = torch.as_tensor(candidate_local, dtype=torch.float32)
    if query_local.ndim != 2 or candidate_local.ndim != 2:
        raise ValueError("Query and candidate local descriptors must be 2D tensors.")
    q_norm = query_local / (query_local.norm(dim=1, keepdim=True) + 1e-8)
    c_norm = candidate_local / (candidate_local.norm(dim=1, keepdim=True) + 1e-8)
    similarity = q_norm @ c_norm.T
    if similarity.numel() == 0:
        return torch.empty((0, 2), dtype=torch.long)
    query_to_candidate = similarity.argmax(dim=1)
    candidate_to_query = similarity.argmax(dim=0)
    n_queries = query_local.size(0)
    query_ids = torch.arange(n_queries, device=similarity.device)
    mutual = candidate_to_query[query_to_candidate] == query_ids
    valid_scores = similarity[query_ids, query_to_candidate][mutual]
    keep = valid_scores > threshold
    retained = query_ids[mutual][keep]
    matched_candidates = query_to_candidate[mutual][keep]
    # return a pair list (query_index, candidate_index)
    if retained.numel() == 0:
        return torch.empty((0, 2), dtype=torch.long, device=similarity.device)
    return torch.stack([retained, matched_candidates], dim=1).long()


In [ ]:
%%writefile src/effovpr/utils/model_artifact.py
"""Complete, offline EffoVPR distribution artifacts; checkpoints remain training-only."""
from __future__ import annotations

import json
from pathlib import Path

import torch
import transformers
from PIL import Image
from safetensors.torch import load_file, save_file

from ..models.effovpr import EffoVPR
from ..models.dino_attention import attention_patch_scores


def load_model_artifact(model_dir, device="cpu"):
    model_dir = Path(model_dir)
    config = json.loads((model_dir / "config.json").read_text(encoding="utf-8"))
    if config["format_version"] != 1 or config["architecture"] != "EffoVPR":
        raise ValueError("Unsupported EffoVPR artifact format")
    model = EffoVPR(**config["model"], backbone_config=config["backbone_config"],
                    processor_config=config["processor_config"])
    if model.backbone.hidden_dim != config["hidden_dim"] or model.backbone.patch_size != config["patch_size"]:
        raise ValueError("Artifact architecture metadata mismatch")
    model.load_state_dict(load_file(str(model_dir / "model.safetensors")), strict=True)
    return model.to(device).eval(), config


def prepare_image(model, image, resolution):
    if isinstance(image, (str, Path)):
        with Image.open(image) as opened:
            image = opened.convert("RGB")
    if resolution <= 0 or resolution % model.backbone.patch_size:
        raise ValueError("Resolution must be a positive multiple of patch size")
    image = image.convert("RGB").resize((resolution, resolution), Image.Resampling.BICUBIC)
    return model.backbone.image_processor(images=[image], do_resize=False,
        do_center_crop=False, return_tensors="pt")["pixel_values"].to(next(model.parameters()).device)


@torch.inference_mode()
def verify_model_artifact(original, model_dir, image):
    restored, config = load_model_artifact(model_dir, next(original.parameters()).device)
    original.eval()
    pixels = prepare_image(original, image, config["inference"]["resolution"])
    restored_pixels = prepare_image(restored, image, config["inference"]["resolution"])
    torch.testing.assert_close(pixels, restored_pixels, rtol=0, atol=0)
    first = original.extract_global(pixels)
    second = restored.extract_global(restored_pixels)
    assert first.shape == second.shape == (1, config["model"]["global_dim"])
    torch.testing.assert_close(first, second, rtol=1e-5, atol=1e-6)
    report = {"global_shape": list(second.shape), "max_absolute_difference": float((first-second).abs().max()),
              "cosine_similarity": float(torch.nn.functional.cosine_similarity(first, second).item())}
    settings = config["reranking"]
    a = original.extract_qkv(pixels, settings["layer"])
    b = restored.extract_qkv(restored_pixels, settings["layer"])
    for key in ("q", "k", "v", "q_patch", "k_patch", "v_patch"):
        torch.testing.assert_close(a[key], b[key], rtol=1e-5, atol=1e-6)
    scores_a = attention_patch_scores(a["q_patch"][0], a["k"][0, 0])
    scores_b = attention_patch_scores(b["q_patch"][0], b["k"][0, 0])
    mask_a, mask_b = scores_a > settings["T1"], scores_b > settings["T1"]
    assert torch.equal(mask_a, mask_b)
    facet = settings["facet"].lower() + "_patch"
    torch.testing.assert_close(a[facet][0, mask_a], b[facet][0, mask_b], rtol=1e-5, atol=1e-6)
    report.update(local_patch_shape=list(b[facet].shape), selected_local_count=int(mask_b.sum()),
                  local_max_absolute_difference=float((a[facet]-b[facet]).abs().max()), strict_load=True)
    return report


def export_best_model(model, checkpoint_path, output_dir, configuration, verification_image):
    """Load the full best state, export every parameter/buffer, and verify reload."""
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if "model_state_dict" not in checkpoint:
        raise ValueError("Export requires a full best model_state_dict; trainable-only checkpoints are not supported")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    config = {
        "format_version": 1, "architecture": "EffoVPR",
        "model": {"backbone_name": model.backbone.model_name, "global_dim": model.global_dim,
            "num_classes": model.num_classes, "fine_tune_last_n_layers": model.fine_tune_last_n_layers,
            "with_registers": model.backbone.with_registers,
            "cosface_scale": model.cosface.scale if model.cosface else 30.0,
            "cosface_margin": model.cosface.margin if model.cosface else 0.4},
        "backbone_config": model.backbone.model.config.to_dict(),
        "processor_config": model.backbone.image_processor.to_dict(),
        "hidden_dim": model.backbone.hidden_dim, "patch_size": model.backbone.patch_size,
        "inference": configuration["inference"], "reranking": configuration["reranking"],
        "class_mapping": checkpoint.get("class_mapping"),
        "source": {"checkpoint": Path(checkpoint_path).name, "best_metric": checkpoint.get("best_metric")},
        "transformers_version": transformers.__version__,
    }
    # clone also breaks shared storage, while retaining every state_dict entry.
    state = {name: tensor.detach().cpu().contiguous().clone() for name, tensor in checkpoint["model_state_dict"].items()}
    save_file(state, str(output_dir / "model.safetensors"), metadata={"format": "pt"})
    del state, checkpoint
    (output_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    report = verify_model_artifact(model, output_dir, verification_image)
    (output_dir / "README.md").write_text(
        "# EffoVPR\n\nFull best-checkpoint weights, including frozen backbone, projection and classifier.\n"
        "Architecture lives in the project Python source. No pretrained download is needed.\n"
        f"Use transformers {transformers.__version__}, PyTorch and safetensors.\n\n"
        "```python\nfrom src.effovpr.utils.model_artifact import load_model_artifact, prepare_image\n"
        "import torch\nmodel, config = load_model_artifact('artifacts/model')\n"
        "pixels = prepare_image(model, '/path/query.jpg', config['inference']['resolution'])\n"
        "with torch.inference_mode():\n    embedding = model.extract_global(pixels)\n"
        "    qkv = model.extract_qkv(pixels, config['reranking']['layer'])\n```\n\n"
        "Use_Model.ipynb accepts query/gallery file paths, retrieves Top 10 with global cosine similarity, "
        "then reranks those candidates by local MNN count (ties retain global rank).\n"
        "Download config.json and model.safetensors as separate files; do not zip the model.\n\n"
        "Verification against best checkpoint:\n```json\n" + json.dumps(report, indent=2) + "\n```\n",
        encoding="utf-8")
    print(json.dumps({"model_dir": str(output_dir), "safetensors_exists": (output_dir / "model.safetensors").is_file(),
                      "verification": report}, indent=2))
    return report


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import faiss
import matplotlib.pyplot as plt

from PIL import Image
from datasets import load_dataset

from src.effovpr.models.effovpr import EffoVPR
from src.effovpr.models.dino_attention import attention_patch_scores
from src.effovpr.reranking.mnn import mutual_nearest_neighbor_matches

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Read the model artifact


In [ ]:
MODEL_DIRECTORY = Path(MODEL_DIR).expanduser().resolve()
for name in ("config.json", "model.safetensors"):
    if not (MODEL_DIRECTORY / name).is_file():
        raise FileNotFoundError(MODEL_DIRECTORY / name)


In [ ]:
artifact_config = json.loads((MODEL_DIRECTORY / "config.json").read_text(encoding="utf-8"))
INFERENCE_RESOLUTION = artifact_config["inference"]["resolution"]
rerank_config = artifact_config["reranking"]
LOCAL_LAYER, FACET = rerank_config["layer"], rerank_config["facet"]
T1, T2 = rerank_config["T1"], rerank_config["T2"]


## Reconstruct architecture and strictly load all weights offline


In [ ]:
from src.effovpr.utils.model_artifact import load_model_artifact
model, artifact_config = load_model_artifact(MODEL_DIRECTORY, DEVICE)
print(artifact_config["model"])


In [ ]:
print("Model ready: config.json + model.safetensors")


## Load query and gallery from file paths


In [ ]:
def load_query_image(path):
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Query image không tồn tại: {path}")
    return Image.open(path).convert("RGB")

query_image = load_query_image(QUERY_IMAGE_PATH)
print("Query:", QUERY_IMAGE_PATH, query_image.size)
display(query_image.resize((320, 320)))

In [ ]:
def load_gallery():
    if GALLERY_SOURCE.lower() == "folder":
        root = Path(GALLERY_DIR).expanduser()
        if not root.is_dir():
            raise FileNotFoundError(f"GALLERY_DIR không tồn tại: {root}")

        extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
        paths = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
        records = []
        for p in paths:
            try:
                image = Image.open(p).convert("RGB")
            except Exception as exc:
                print("Skip invalid image:", p, exc)
                continue
            records.append({
                "id": str(p.relative_to(root)),
                "path": str(p.resolve()),
                "image": image,
                "label": "",
                "dataset_index": len(records),
            })
        if not records:
            raise ValueError(f"Không có ảnh hợp lệ trong {root}")
        return records

    raise ValueError("GALLERY_SOURCE phải là 'dataset' hoặc 'folder'.")

gallery = load_gallery()
print(f"Gallery images: {len(gallery):,}")
print("First IDs:", [r["id"] for r in gallery[:5]])

## 6. Global retrieval — lấy đúng Top 10 candidate

Đây là bước chọn candidate trước local reranking:

`query → global embedding → FAISS → top 10`

Sau đó **chỉ 10 ảnh này** mới được đưa vào MNN reranking.

In [ ]:
def preprocess_images(model, images, resolution):
    if resolution <= 0 or resolution % model.backbone.patch_size != 0:
        raise ValueError(
            f"Resolution phải là bội số của patch size {model.backbone.patch_size}."
        )

    resized = [
        image.convert("RGB").resize(
            (resolution, resolution),
            resample=Image.Resampling.BICUBIC,
        )
        for image in images
    ]

    processed = model.backbone.image_processor(
        images=resized,
        do_resize=False,
        do_center_crop=False,
        return_tensors="pt",
    )
    return processed["pixel_values"].to(DEVICE)


@torch.inference_mode()
def extract_gallery_features(model, records, resolution, batch_size):
    all_features = []

    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        pixels = preprocess_images(
            model,
            [r["image"] for r in batch],
            resolution,
        )
        features = model.extract_global(pixels)
        all_features.append(features.float().cpu().numpy())

        if (start // batch_size) % 10 == 0:
            print(f"Global gallery: {min(start + batch_size, len(records)):,}/{len(records):,}")

    return np.concatenate(all_features, axis=0).astype(np.float32)


@torch.inference_mode()
def extract_query_global(model, image, resolution):
    pixels = preprocess_images(model, [image], resolution)
    return model.extract_global(pixels).float().cpu().numpy()


gallery_features = extract_gallery_features(
    model,
    gallery,
    INFERENCE_RESOLUTION,
    GLOBAL_BATCH_SIZE,
)

print("Gallery feature matrix:", gallery_features.shape)

index = faiss.IndexFlatIP(gallery_features.shape[1])
index.add(gallery_features)

query_feature = extract_query_global(
    model,
    query_image,
    INFERENCE_RESOLUTION,
)

global_scores, global_indices = index.search(
    query_feature,
    min(RERANK_TOP_K, len(gallery)),
)

global_scores = global_scores[0]
global_indices = global_indices[0]

print("Global Top-K:", len(global_indices))

In [ ]:
global_candidates = []

for rank, (score, idx) in enumerate(zip(global_scores, global_indices), start=1):
    record = gallery[int(idx)]
    global_candidates.append({
        "global_rank": rank,
        "gallery_index": int(idx),
        "image_path": record["path"],
        "image_id": record["id"],
        "label": record.get("label", ""),
        "global_score": float(score),
        "image": record["image"],
    })

global_table = pd.DataFrame([
    {
        "global_rank": c["global_rank"],
        "image_path": record["path"],
        "image_id": c["image_id"],
        "label": c["label"],
        "global_score": c["global_score"],
    }
    for c in global_candidates
])

display(global_table)

## 7. Local reranking — MNN trên Top 10

Với mỗi candidate:

1. Lấy Q/K/V ở layer `n-1`.
2. Tính attention score của patch đối với CLS.
3. Giữ patch có score `> T1`.
4. Lấy facet `V`.
5. Tính Mutual Nearest Neighbor với ngưỡng `T2`.
6. Số MNN matches là reranking score.

Thứ tự cuối cùng: **MNN score giảm dần**, nếu bằng nhau thì giữ thứ tự global ban đầu.

In [ ]:
@torch.inference_mode()
def extract_local_descriptors(model, image, resolution, layer, facet, threshold):
    pixels = preprocess_images(model, [image], resolution)
    qkv = model.extract_qkv(pixels, layer=layer)

    scores = attention_patch_scores(
        qkv["q_patch"][0],
        qkv["k"][0, 0],
    )

    keep = scores > threshold
    descriptors = qkv[f"{facet.lower()}_patch"][0, keep].float()

    return descriptors


query_local = extract_local_descriptors(
    model,
    query_image,
    INFERENCE_RESOLUTION,
    LOCAL_LAYER,
    FACET,
    T1,
)

print("Query selected local descriptors:", tuple(query_local.shape))

In [ ]:
reranked_candidates = []

for candidate in global_candidates:
    candidate_local = extract_local_descriptors(
        model,
        candidate["image"],
        INFERENCE_RESOLUTION,
        LOCAL_LAYER,
        FACET,
        T1,
    )

    matches = mutual_nearest_neighbor_matches(
        query_local,
        candidate_local,
        threshold=T2,
    )

    candidate_result = {
        **candidate,
        "mnn_matches": int(matches.shape[0]),
        "local_descriptor_count": int(candidate_local.shape[0]),
    }
    reranked_candidates.append(candidate_result)

# Same tie-breaking principle as the original evaluation code:
# higher MNN first, then earlier global rank.
reranked_candidates.sort(
    key=lambda item: (-item["mnn_matches"], item["global_rank"])
)

for final_rank, candidate in enumerate(reranked_candidates, start=1):
    candidate["final_rank"] = final_rank

final_table = pd.DataFrame([
    {
        "final_rank": c["final_rank"],
        "image_id": c["image_id"],
        "label": c["label"],
        "MNN_matches": c["mnn_matches"],
        "global_rank": c["global_rank"],
        "global_score": c["global_score"],
        "local_descriptors": c["local_descriptor_count"],
    }
    for c in reranked_candidates
])

display(final_table)

## 8. Kết quả cuối cùng + hiển thị Top 10

`final_rank = 1` là kết quả cuối cùng được chọn sau reranking.

Nếu gallery là dataset hoặc folder ảnh, notebook có thể hiển thị trực tiếp 10 ảnh candidate.

In [ ]:
print("Image path:", reranked_candidates[0]["image_path"])
best = reranked_candidates[0]

print("=" * 80)
print("FINAL RESULT")
print("=" * 80)
print("Image ID :", best["image_id"])
print("Label    :", best["label"])
print("MNN      :", best["mnn_matches"])
print("Global rank:", best["global_rank"])
print("Global score:", best["global_score"])
print("=" * 80)

# Hiển thị query + 10 kết quả
n = len(reranked_candidates)
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
axes = axes.flatten()

axes[0].imshow(query_image)
axes[0].set_title("QUERY")
axes[0].axis("off")

for ax in axes[1:]:
    ax.axis("off")

for i, candidate in enumerate(reranked_candidates[:10], start=1):
    ax = axes[i]
    ax.imshow(candidate["image"])
    title = (
        f"#{candidate['final_rank']} | "
        f"MNN={candidate['mnn_matches']}\n"
        f"ID={candidate['image_id']}"
    )
    ax.set_title(title, fontsize=8)
    ax.axis("off")

plt.tight_layout()

OUTPUT_DIR = WORK_DIR / "artifacts/results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

figure_path = OUTPUT_DIR / "retrieval_top10.png"
plt.savefig(figure_path, dpi=140, bbox_inches="tight")
plt.show()

print("Saved:", figure_path)

In [ ]:
# Lưu kết quả để có thể sử dụng bên ngoài notebook.
result_rows = []
for candidate in reranked_candidates:
    result_rows.append({
        "final_rank": candidate["final_rank"],
        "image_id": candidate["image_id"],
        "label": candidate["label"],
        "mnn_matches": candidate["mnn_matches"],
        "global_rank": candidate["global_rank"],
        "global_score": candidate["global_score"],
        "local_descriptor_count": candidate["local_descriptor_count"],
    })

results_df = pd.DataFrame(result_rows)
csv_path = OUTPUT_DIR / "top10_results.csv"
results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

summary = {
    "query_image": str(Path(QUERY_IMAGE_PATH).resolve()),
    "checkpoint": str(CHECKPOINT.resolve()),
    "gallery_source": GALLERY_SOURCE,
    "gallery_count": len(gallery),
    "rerank_top_k": len(reranked_candidates),
    "resolution": INFERENCE_RESOLUTION,
    "local_layer": LOCAL_LAYER,
    "facet": FACET,
    "T1": T1,
    "T2": T2,
    "final_image_id": best["image_id"],
    "final_label": best["label"],
    "final_mnn_matches": best["mnn_matches"],
}

json_path = OUTPUT_DIR / "result.json"
json_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("CSV :", csv_path)
print("JSON:", json_path)
print("FIG :", figure_path)